# Download readable articles and books

Files are saved directly to `~/Downloads/books_articles`. There is no database and no image download.

- arXiv: search XML and paper PDF
- Europe PMC / PubMed Central: open-access article XML and extracted plain text
- Open Library: readable plain-text books only when the search result has a public Internet Archive scan

The code keeps SSL certificate verification enabled.

In [7]:
from pathlib import Path
import time
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen
import json
import re
import ssl
import xml.etree.ElementTree as ET

try:
    import certifi
except ImportError as error:
    raise RuntimeError('Missing certificate bundle. Run: %pip install certifi') from error

DOWNLOAD_FOLDER = Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001' / 'a.Extractions' /  'a.Raw Extraction Documents - Regional Status tool'
DOWNLOAD_FOLDER.mkdir(parents=True, exist_ok=True)
USER_AGENT = 'BooksArticlesDownloader/0.5 (contact: your-email@example.com)'
SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())

# 60s was too generous for what should be fast JSON/text responses -- a single
# slow/stalled request could block a loop for a full minute. 15s is enough
# headroom for a normal response while failing fast on a stalled connection.
REQUEST_TIMEOUT_SECONDS = 15

def safe_filename(text: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', text).strip('._')[:120]

def download(url: str, filename: str, params: dict | None = None, quiet: bool = False) -> Path:
    if params:
        url += ('&' if '?' in url else '?') + urlencode(params)
    destination = DOWNLOAD_FOLDER / filename
    request = Request(url, headers={'User-Agent': USER_AGENT})
    with urlopen(request, context=SSL_CONTEXT, timeout=REQUEST_TIMEOUT_SECONDS) as response, destination.open('wb') as file:
        while chunk := response.read(1024 * 1024):
            file.write(chunk)
    if not quiet:
        print(f'Saved: {destination}')
    return destination

def read_json(url: str, params: dict) -> dict:
    request = Request(url + '?' + urlencode(params), headers={'User-Agent': USER_AGENT, 'Accept': 'application/json'})
    with urlopen(request, context=SSL_CONTEXT, timeout=REQUEST_TIMEOUT_SECONDS) as response:
        return json.load(response)

def xml_to_text(xml_file: Path, text_file: Path) -> Path:
    root = ET.parse(xml_file).getroot()
    text = '\n'.join(line.strip() for line in root.itertext() if line.strip())
    text_file.write_text(text, encoding='utf-8')
    print(f'Saved: {text_file}')
    return text_file

def write_source_metadata(text_path: Path, **fields) -> None:
    """Write a sidecar '<text_path>.meta.json' capturing real citation data
    resolved during download (title, url, isbn, doi, etc.) -- so downstream
    RAG ingestion (04_ChunkEmbedChromaRetrieve) can build proper citations
    instead of guessing from the filename.

    Only pass fields you actually know are true; leave unknown ones out
    (or None) rather than guessing -- a wrong citation is worse than a
    missing one.
    """
    meta_path = text_path.with_suffix(text_path.suffix + '.meta.json')
    payload = {k: v for k, v in fields.items() if v is not None}
    meta_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


In [8]:
# Corpus manisfest
from corpus_manifest import BOOKS, SPECIES_WIKI, REGIONAL_STATUS_SOURCES, PAPERS, ALL_DOCUMENTS, DOCUMENT_GROUPS

# Count the number of documents listed
print(f"ALL_DOCUMENTS {len(ALL_DOCUMENTS)}")
for label, docs in DOCUMENT_GROUPS.items():
    print(label, len(docs))

# Get the title of documents
def get_unique_values_titles(documents, key='title'):
    return [doc[key] for doc in documents]

REQUEST_DELAY_SECONDS = 1.0  # politeness delay between API calls in each loop below

# Report titles no found
def report(api_name: str, status: str, detail: str) -> None:
    """Consistent status line for every extraction loop, e.g. '[arXiv] NO MATCH: <title>'."""
    print(f'[{api_name}] {status}: {detail}')


ALL_DOCUMENTS 362
species_wiki 245
regional_status 32
books 47
papers 27
videos 11


In [9]:
# Europe PMC / PubMed Central: search by title for each paper in PAPERS,
# save XML + extracted TXT for the first open-access full-text match.
# Fallback: if no open-access full-text article exists, save the abstract
# instead (still a useful RAG snippet, just clearly marked as abstract-only).

API_NAME = 'Europe PMC'
paper_titles_pmc = get_unique_values_titles(PAPERS)

for title in paper_titles_pmc:
    try:
        pmc_results = read_json(
            'https://www.ebi.ac.uk/europepmc/webservices/rest/search',
            {'query': f'TITLE:"{title}" AND OPEN_ACCESS:Y', 'format': 'json', 'pageSize': 5, 'resultType': 'core'},
        )
        article = next((item for item in pmc_results.get('resultList', {}).get('result', []) if item.get('pmcid')), None)

        if article is not None:
            pmcid = article['pmcid']
            text_path = DOWNLOAD_FOLDER / f'europe_pmc_{pmcid}_fulltext.txt'
            if text_path.exists():
                report(API_NAME, 'SKIP', text_path.name)
                continue
            article_xml = download(
                f'https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML',
                f'europe_pmc_{pmcid}_fulltext.xml',
            )
            xml_to_text(article_xml, text_path)
            # article fields (resultType=core) come straight from Europe PMC's
            # own bibliographic record -- genuine data, not a guess.
            write_source_metadata(
                text_path,
                source_type='europe_pmc',
                title=article.get('title') or title,
                author=article.get('authorString'),
                year=article.get('pubYear'),
                journal=article.get('journalTitle'),
                doi=article.get('doi'),
                pmcid=pmcid,
                pmid=article.get('pmid'),
                url=f'https://europepmc.org/article/PMC/{pmcid}',
                full_text=True,
            )
            continue

        # Fallback: no open-access full text -- try to save the abstract instead.
        fallback_results = read_json(
            'https://www.ebi.ac.uk/europepmc/webservices/rest/search',
            {'query': f'TITLE:"{title}"', 'format': 'json', 'pageSize': 5, 'resultType': 'core'},
        )
        fallback_article = next(
            (item for item in fallback_results.get('resultList', {}).get('result', []) if item.get('abstractText')),
            None,
        )
        if fallback_article is None:
            report(API_NAME, 'NO MATCH', title)
            continue

        record_id = fallback_article.get('pmid') or fallback_article.get('id') or safe_filename(title)
        abstract_path = DOWNLOAD_FOLDER / f'europe_pmc_{record_id}_abstract.txt'
        if abstract_path.exists():
            report(API_NAME, 'SKIP', abstract_path.name)
            continue

        abstract_path.write_text(fallback_article['abstractText'], encoding='utf-8')
        write_source_metadata(
            abstract_path,
            source_type='europe_pmc',
            title=fallback_article.get('title') or title,
            author=fallback_article.get('authorString'),
            year=fallback_article.get('pubYear'),
            journal=fallback_article.get('journalTitle'),
            doi=fallback_article.get('doi'),
            pmid=fallback_article.get('pmid'),
            url=f"https://europepmc.org/abstract/MED/{fallback_article.get('pmid', '')}" if fallback_article.get('pmid') else None,
            full_text=False,
        )
        report(API_NAME, 'FALLBACK', f'{title} -> saved abstract only (no open-access full text)')
    except Exception as error:
        report(API_NAME, 'FAILED', f'{title} ({error})')
    time.sleep(REQUEST_DELAY_SECONDS)


[Europe PMC] NO MATCH: Biological Flora of the British Isles: Fallopia japonica
[Europe PMC] NO MATCH: Impacts of Himalayan balsam (Impatiens glandulifera) on riparian plant communities in the UK
[Europe PMC] NO MATCH: A review of the ecology and control of Rhododendron ponticum
[Europe PMC] NO MATCH: Genetics of hybridisation between native bluebell (Hyacinthoides non-scripta) and Spanish bluebell (H. hispanica)
[Europe PMC] NO MATCH: Coniine and other poisonous alkaloids in Conium maculatum (hemlock): chemistry and toxicology
[Europe PMC] NO MATCH: Cardiac glycosides in Digitalis purpurea (foxglove): biosynthesis and pharmacological history
[Europe PMC] NO MATCH: Tropane alkaloid poisoning by Atropa belladonna and Datura stramonium: clinical review
[Europe PMC] SKIP: europe_pmc_PMC11640968_fulltext.txt
[Europe PMC] NO MATCH: Molecular phylogeny and reclassification of the genus Amanita in Europe
[Europe PMC] NO MATCH: Orellanine poisoning from Cortinarius rubellus and C. orellanus: d